# Phase 2 Services — Interactive Validation

This notebook exercises every Phase 2 service against a real SQLite database.
No LLM calls are made — `ResumeParser` runs in heuristic-only mode (`enhance_fn=None`).

Run each section in order. Every cell ends with assertions so failures surface immediately.

**Sections**
1. Setup & imports
2. JobPosting and ResumeProfile schemas
3. SkillNormalizer
4. StatusManager — workflow transitions
5. StatusManager — job transitions
6. JobDiscoveryService — normalize and deduplicate
7. ResumeParser — heuristic parse and caching
8. ObservabilityService — full event chain
9. ReportGenerator — run summary and job report
10. Cleanup

---
## 1. Setup & Imports

**Purpose:** Establish an isolated test environment with a fresh SQLite database.

**Why each import:**
- `os`, `sys`, `Path` — set up project root so all `app.*` imports resolve correctly regardless of where Jupyter was launched from
- `uuid` — generate stable IDs for test data
- `MagicMock` — mock the v1 scraper interface so `JobDiscoveryService` can be tested without real network calls
- `TEMP_DB` — all writes go here, never to `data/v2.db`; deleted on cleanup

**What it proves:** The project is importable and the test DB path is isolated.

In [ ]:
import os
import sys
import uuid
from pathlib import Path
from unittest.mock import MagicMock

# Resolve project root — works whether Jupyter launched from root or from notebooks/
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

TEMP_DB = Path('data/notebook_p2_test.db')
if TEMP_DB.exists():
    TEMP_DB.unlink()

# Initialise the full Phase 1 schema (all 18 tables + Phase 2 raw_text_hash migration)
from app.repositories.database import init_db, utcnow_iso
init_db(TEMP_DB)

print(f'Project root : {project_root}')
print(f'Test DB      : {TEMP_DB}')
print('Database initialised.')

---
## 2. JobPosting and ResumeProfile Schemas

**Purpose:** Verify the two new Phase 2 data schemas are correctly validated by Pydantic.

**What it proves:**
- `JobPosting` defaults `work_mode` to `UNKNOWN` when the posting does not specify it
- `SalaryInfo` allows all fields to be `None` — missing salary is valid
- `ResumeProfile` rejects empty `raw_text` — enforcing the fidelity anchor at schema level
- `ExperienceEntry.end_year = None` correctly represents a current role

In [ ]:
from pydantic import ValidationError

from app.schemas.job_posting import JobPosting, JobSource, SalaryInfo, WorkMode
from app.schemas.resume_profile import (
    CertificationEntry, EducationEntry, ExperienceEntry, ResumeProfile,
)

NOW = utcnow_iso()

# --- JobPosting ---
jp = JobPosting(
    job_id=str(uuid.uuid4()),
    workflow_id='wf-nb-001',
    url='https://linkedin.com/jobs/1',
    source=JobSource.LINKEDIN,
    title='Staff Engineer',
    company='TechCorp',
    location='Remote',
    found_at=NOW,
)
assert jp.work_mode == WorkMode.UNKNOWN, 'work_mode should default to UNKNOWN'
assert jp.salary is None
print(f'JobPosting: {jp.title} @ {jp.company}, work_mode={jp.work_mode.value}')

# work_mode with explicit value
jp_remote = JobPosting(**{**jp.model_dump(), 'work_mode': WorkMode.REMOTE})
assert jp_remote.work_mode == WorkMode.REMOTE
print(f'  explicit work_mode=remote: {jp_remote.work_mode.value}')

# SalaryInfo — all None is valid
empty_salary = SalaryInfo()
assert empty_salary.min_amount is None
print(f'SalaryInfo (all None): {empty_salary.model_dump()}')

# --- ResumeProfile ---
resume_text = 'Jane Smith\njane@example.com\n\nSKILLS\nPython, Kubernetes, GCP\n' * 5
rp = ResumeProfile(
    resume_id=str(uuid.uuid4()),
    raw_text=resume_text,
    parsed_at=NOW,
    name='Jane Smith',
    skills=['Python', 'Kubernetes', 'GCP'],
    experience=[
        ExperienceEntry(company='TechCorp', title='Staff Engineer', start_year=2020),
    ],
)
assert rp.experience[0].end_year is None, 'end_year=None means current role'
print(f'ResumeProfile: {rp.name}, {len(rp.skills)} skills')
print(f'  experience[0].end_year={rp.experience[0].end_year} (current role)')

# Reject empty raw_text — the fidelity anchor must always be set
try:
    ResumeProfile(resume_id='bad', raw_text='', parsed_at=NOW)
    assert False, 'Should have raised'
except ValidationError:
    print('✓ ResumeProfile rejects empty raw_text')

print('\n✓ JobPosting and ResumeProfile schemas validated')

---
## 3. SkillNormalizer

**Purpose:** Verify that raw skill strings from resumes and job descriptions are mapped to
canonical names using `data/skills.yaml`.

**Why this matters:** Agents compare candidate skills against job requirements.
Without normalization, `'k8s'` and `'Kubernetes'` look like different skills —
causing false gaps in scoring.

**What it proves:**
- Case-insensitive alias lookup works end-to-end from the YAML file
- Unknown skills pass through unchanged — no silent data loss
- `normalize_and_deduplicate` collapses all aliases to one canonical entry

In [ ]:
from app.services.skill_normalizer import SkillNormalizer

sn = SkillNormalizer()  # loads data/skills.yaml

test_cases = [
    ('python',   'Python'),
    ('python3',  'Python'),
    ('PYTHON',   'Python'),
    ('k8s',      'Kubernetes'),
    ('K8s',      'Kubernetes'),
    ('aws',      'AWS'),
    ('gcp',      'GCP'),
    ('google cloud', 'GCP'),
    ('tf',       'Terraform'),
    ('grpc',     'gRPC'),
    ('AWS',      'AWS'),  # already canonical
]

print('Single normalize() lookups:')
for raw, expected in test_cases:
    result = sn.normalize(raw)
    status = '✓' if result == expected else '✗'
    print(f'  {status} normalize({raw!r}) = {result!r}  (expected {expected!r})')
    assert result == expected, f'Expected {expected!r}, got {result!r}'

# Unknown skill passes through
unknown = sn.normalize('cobol_2000')
assert unknown == 'cobol_2000'
print(f'\n  ✓ Unknown skill passthrough: normalize("cobol_2000") = {unknown!r}')

# normalize_and_deduplicate — three aliases → one canonical entry
raw_skills = ['python', 'Python', 'python3', 'k8s', 'Kubernetes']
deduped = sn.normalize_and_deduplicate(raw_skills)
assert 'Python' in deduped
assert 'Kubernetes' in deduped
assert deduped.count('Python') == 1
assert deduped.count('Kubernetes') == 1
print(f'\nnormalize_and_deduplicate({raw_skills})')
print(f'  → {deduped}')
print('  ✓ 5 raw inputs → 2 canonical entries')

print('\n✓ SkillNormalizer validated')

---
## 4. StatusManager — Workflow Transitions

**Purpose:** Confirm that workflow status transitions are validated correctly.

**Why this matters:** The orchestrator calls `StatusManager.transition_workflow()` before
every status write. An invalid transition (e.g., `completed → running`) must be rejected
before it reaches the DB — once a terminal state is written, the workflow cannot be resumed.

**What it proves:**
- All valid transitions succeed and return the new status
- All terminal → any transitions raise `InvalidTransitionError` with context
- `is_terminal_workflow()` correctly identifies the three terminal states

In [ ]:
from app.services.status_manager import InvalidTransitionError, StatusManager
from app.state.workflow_state import WorkflowStatus

sm = StatusManager()
WF_ID = 'wf-nb-001'

# Valid transitions
valid_transitions = [
    (WorkflowStatus.INITIALIZED,     WorkflowStatus.RUNNING),
    (WorkflowStatus.RUNNING,         WorkflowStatus.WAITING_FOR_USER),
    (WorkflowStatus.WAITING_FOR_USER, WorkflowStatus.RUNNING),
    (WorkflowStatus.RUNNING,         WorkflowStatus.COMPLETED),
    (WorkflowStatus.RUNNING,         WorkflowStatus.FAILED),
    (WorkflowStatus.RUNNING,         WorkflowStatus.CANCELLED),
]

print('Valid workflow transitions:')
for current, new in valid_transitions:
    result = sm.transition_workflow(WF_ID, current, new)
    assert result == new
    print(f'  ✓ {current.value} → {new.value}')

# Invalid transitions — terminal states are locked
invalid_transitions = [
    (WorkflowStatus.COMPLETED, WorkflowStatus.RUNNING),
    (WorkflowStatus.FAILED,    WorkflowStatus.RUNNING),
    (WorkflowStatus.CANCELLED, WorkflowStatus.RUNNING),
]

print('\nInvalid transitions (must raise):')
for current, new in invalid_transitions:
    try:
        sm.transition_workflow(WF_ID, current, new)
        print(f'  ✗ {current.value} → {new.value} should have raised')
    except InvalidTransitionError as e:
        assert WF_ID in str(e)
        print(f'  ✓ {current.value} → {new.value} raised InvalidTransitionError')

# Terminal checks
print('\nTerminal status checks:')
for status in [WorkflowStatus.COMPLETED, WorkflowStatus.FAILED, WorkflowStatus.CANCELLED]:
    assert sm.is_terminal_workflow(status)
    print(f'  ✓ is_terminal({status.value}) = True')
for status in [WorkflowStatus.RUNNING, WorkflowStatus.WAITING_FOR_USER]:
    assert not sm.is_terminal_workflow(status)
    print(f'  ✓ is_terminal({status.value}) = False')

print('\n✓ StatusManager workflow transitions validated')

---
## 5. StatusManager — Job Transitions

**Purpose:** Confirm the full `JobStatus` state machine — introduced in Phase 2.

**Why this matters:** `JobStatus` tracks where each job is in the pipeline
(`DISCOVERED → SCORED → SHORTLISTED → REVIEWED → APPLIED → REJECTED/OFFER/PASSED`).
Invalid jumps (e.g., skipping scoring) must be caught before state is written.

**What it proves:**
- The happy path (discover → score → shortlist → review → apply → offer) all succeed
- Both `PASSED` exit points work (user skips after scoring or after review)
- Terminal job statuses cannot be transitioned further

In [ ]:
from app.services.status_manager import JobStatus

JOB_ID = 'job-nb-001'

# Happy path
happy_path = [
    (JobStatus.DISCOVERED,  JobStatus.SCORED),
    (JobStatus.SCORED,      JobStatus.SHORTLISTED),
    (JobStatus.SHORTLISTED, JobStatus.REVIEWED),
    (JobStatus.REVIEWED,    JobStatus.APPLIED),
    (JobStatus.APPLIED,     JobStatus.OFFER),
]

print('Happy path (discover → offer):')
for current, new in happy_path:
    result = sm.transition_job(JOB_ID, current, new)
    assert result == new
    print(f'  ✓ {current.value} → {new.value}')

# Exit via PASSED
print('\nPASSED exit points:')
for current in [JobStatus.SCORED, JobStatus.REVIEWED]:
    result = sm.transition_job(JOB_ID, current, JobStatus.PASSED)
    assert result == JobStatus.PASSED
    print(f'  ✓ {current.value} → passed')

# Invalid transition
print('\nInvalid transitions (must raise):')
invalid_job_transitions = [
    (JobStatus.SHORTLISTED, JobStatus.DISCOVERED),
    (JobStatus.OFFER,       JobStatus.APPLIED),
    (JobStatus.PASSED,      JobStatus.SCORED),
]
for current, new in invalid_job_transitions:
    try:
        sm.transition_job(JOB_ID, current, new)
        print(f'  ✗ {current.value} → {new.value} should have raised')
    except InvalidTransitionError as e:
        assert JOB_ID in str(e)
        print(f'  ✓ {current.value} → {new.value} raised InvalidTransitionError')

# Terminal checks
print('\nTerminal job status checks:')
for status in [JobStatus.PASSED, JobStatus.REJECTED, JobStatus.OFFER]:
    assert sm.is_terminal_job(status)
    print(f'  ✓ is_terminal({status.value}) = True')

print('\n✓ StatusManager job transitions validated')

---
## 6. JobDiscoveryService — Normalize and Deduplicate

**Purpose:** Verify the v1 → v2 normalization pipeline and URL-based deduplication.

**Why this matters:** Every job in v2 must be a `JobPosting` — v1 `Job` types must not
leak past this service. Deduplication prevents the same job from appearing twice when
multiple scrapers return it, or when the scraper re-runs on a subsequent workflow run.

**What it proves:**
- `normalize()` converts mock v1 Jobs to `JobPosting` with correct field mapping
- `deduplicate()` drops within-batch URL duplicates and DB-existing URLs
- `discover()` applies the title filter and caps results at `max_jobs`
- Scraper failures are swallowed — partial results are returned

In [ ]:
from app.repositories.job_repository import JobRepository
from app.schemas.job_posting import JobSource, WorkMode
from app.services.job_discovery_service import JobDiscoveryService

job_repo = JobRepository(TEMP_DB)

def make_v1_job(url, title='Staff Engineer', company='Acme', source='linkedin', work_mode=None):
    j = MagicMock()
    j.url = url
    j.title = title
    j.company = company
    j.source = source
    j.work_mode = work_mode
    j.location = 'Remote'
    j.description = 'A great role for a Staff Engineer.'
    j.salary = None
    j.found_at = None
    j.posted_at = None
    return j

config = {'search': {'max_jobs': 20}}
svc = JobDiscoveryService(job_repository=job_repo, config=config)

# normalize() — v1 Job → JobPosting
v1_job = make_v1_job('https://linkedin.com/jobs/100', work_mode='remote')
posting = svc.normalize(v1_job, 'wf-nb-001')
assert posting.source == JobSource.LINKEDIN
assert posting.work_mode == WorkMode.REMOTE
assert posting.workflow_id == 'wf-nb-001'
assert len(posting.job_id) == 36  # UUID
print(f'normalize(): source={posting.source.value}, work_mode={posting.work_mode.value}')
print(f'  job_id={posting.job_id}')

# Work mode None → UNKNOWN
posting_no_mode = svc.normalize(make_v1_job('https://example.com/2'), 'wf-nb-001')
assert posting_no_mode.work_mode == WorkMode.UNKNOWN
print(f'  work_mode=None → {posting_no_mode.work_mode.value}')

# deduplicate() — batch duplicates
jobs = [
    svc.normalize(make_v1_job('https://example.com/dup'), 'wf'),
    svc.normalize(make_v1_job('https://example.com/dup'), 'wf'),  # same URL
    svc.normalize(make_v1_job('https://example.com/unique'), 'wf'),
]
deduped = svc.deduplicate(jobs)
assert len(deduped) == 2
print(f'\ndeduplicate() — 3 jobs (1 URL dup) → {len(deduped)} unique')

# discover() — caps results
scraper = MagicMock()
scraper.scrape.return_value = [
    make_v1_job(f'https://example.com/j{i}') for i in range(30)
]
capped_svc = JobDiscoveryService(job_repo, {'search': {'max_jobs': 5}}, scrapers=[scraper])
results = capped_svc.discover('wf-nb-001', {})
assert len(results) <= 5
print(f'\ndiscover() — 30 scraped, max_jobs=5 → {len(results)} returned')

# discover() — scraper failure is swallowed
bad_scraper = MagicMock()
bad_scraper.scrape.side_effect = RuntimeError('connection refused')
good_scraper = MagicMock()
good_scraper.scrape.return_value = [make_v1_job('https://example.com/good')]
resilient_svc = JobDiscoveryService(job_repo, config, scrapers=[bad_scraper, good_scraper])
partial = resilient_svc.discover('wf-nb-001', {})
assert len(partial) == 1
print(f'discover() — 1 scraper fails, 1 succeeds → {len(partial)} result')

print('\n✓ JobDiscoveryService validated')

---
## 7. ResumeParser — Heuristic Parse and Caching

**Purpose:** Verify heuristic text extraction and SHA-256-based caching against a real DB.

**Why `enhance_fn=None`:** Phase 2 tests have no LLM infrastructure. The heuristic parse
path validates that `raw_text` is always captured and that caching works correctly.
Claude enhancement is wired by the orchestrator in Phase 5.

**What it proves:**
- `parse_text()` extracts email and skills heuristically from a realistic resume fixture
- `raw_text` is always set — the Fidelity Reviewer's source of truth is never lost
- A second parse of the same text returns the cached profile (no DB write)
- Empty text raises `ResumeParseError` before any caching or parsing is attempted
- The `enhance_fn` receives `(raw_text, heuristic_fields)` when provided (mock test)

In [ ]:
from app.repositories.resume_repository import ResumeRepository
from app.services.resume_parser import ResumeParseError, ResumeParser

resume_repo = ResumeRepository(TEMP_DB)
parser = ResumeParser(resume_repository=resume_repo, enhance_fn=None)

RESUME_TEXT = """
Jane Smith
jane.smith@example.com | Atlanta, GA

SUMMARY
Staff Engineer with 12 years building distributed systems at scale.
Led platform migrations serving 50M users. Strong in Python, GCP, Kubernetes.

EXPERIENCE
TechCorp — Staff Engineer (2020–Present)
  Led migration of 18 services to GCP. Introduced Kubernetes-based CI/CD.
  Reduced deployment time from 4 hours to 12 minutes.

DataSys — Senior Engineer (2016–2020)
  Built real-time data pipeline processing 2M events/sec using Kafka and Spark.

EDUCATION
Georgia Institute of Technology — B.S. Computer Science, 2012

SKILLS
Python, Go, Kubernetes, GCP, Kafka, Spark, Terraform, PostgreSQL, Redis

CERTIFICATIONS
Certified Kubernetes Administrator (CKA) — CNCF, 2022
""" * 3  # repeat to exceed 50-char minimum comfortably

# First parse — cache miss, heuristic parse runs
profile = parser.parse_text(RESUME_TEXT, 'jane_smith_resume.pdf')

assert profile.raw_text == RESUME_TEXT, 'raw_text must equal the original input'
assert profile.email == 'jane.smith@example.com'
assert len(profile.resume_id) == 36
assert profile.parsed_at.endswith('Z')
print(f'First parse (cache miss):')
print(f'  resume_id : {profile.resume_id}')
print(f'  email     : {profile.email}')
print(f'  skills    : {profile.skills}')
print(f'  raw_text  : {len(profile.raw_text)} chars')

# Second parse — same text → cache hit, same resume_id returned
profile2 = parser.parse_text(RESUME_TEXT, 'jane_smith_resume.pdf')
assert profile2.resume_id == profile.resume_id, \
    'Cache hit must return the same resume_id'
print(f'\nSecond parse (cache hit):')
print(f'  resume_id : {profile2.resume_id} (same as first parse ✓)')

# Empty text raises ResumeParseError
try:
    parser.parse_text('', 'empty.pdf')
    assert False
except ResumeParseError as e:
    print(f'\nEmpty text → ResumeParseError: {e}')

# enhance_fn mock — called with (raw_text, heuristic_fields), Claude fields take precedence
mock_enhance = MagicMock(return_value={
    'name': 'Jane Smith (enhanced)',
    'skills': ['Python', 'Kubernetes', 'GCP', 'Kafka'],
    'experience': [
        {'company': 'TechCorp', 'title': 'Staff Engineer', 'start_year': 2020},
    ],
})
enhanced_parser = ResumeParser(resume_repository=resume_repo, enhance_fn=mock_enhance)
NEW_TEXT = 'John Doe\njohn@example.com\n\nSKILLS\nPython\n' * 10
enhanced = enhanced_parser.parse_text(NEW_TEXT, 'john_doe.pdf')
mock_enhance.assert_called_once()
assert enhanced.name == 'Jane Smith (enhanced)'  # Claude's value wins
assert enhanced.raw_text == NEW_TEXT             # raw_text is always original
assert len(enhanced.experience) == 1
print(f'\nenhance_fn mock test:')
print(f'  enhance_fn called: {mock_enhance.call_count} time(s)')
print(f'  name (from Claude): {enhanced.name}')
print(f'  raw_text preserved: {len(enhanced.raw_text)} chars')
print(f'  experience entries: {len(enhanced.experience)}')

print('\n✓ ResumeParser validated')

---
## 8. ObservabilityService — Full Event Chain

**Purpose:** Walk through a complete agent execution lifecycle using the service's typed API
and verify all events land in the database.

**Why this matters:** Every agent call in Phase 4 will go through `ObservabilityService`.
The service must correctly wire all four repositories, and must not raise if one fails.

**What it proves:**
- `log_agent_started()` returns an `event_id` UUID and writes to `agent_events`
- `log_agent_completed()` writes a correlated completion event
- `log_llm_call()` writes to `llm_calls`
- `log_step_started()` returns a `step_execution_id` and writes to `step_executions`
- `log_security_event()` writes to `security_events`
- A failing repo does not crash the caller (swallow + log pattern)

In [ ]:
from app.repositories.decision_repository import DecisionRepository
from app.repositories.observability_repository import ObservabilityRepository
from app.repositories.security_repository import SecurityRepository
from app.repositories.step_repository import StepRepository
from app.repositories.database import get_connection
from app.services.observability_service import ObservabilityService
from app.state.workflow_state import WorkflowStep

OBS_WF_ID = 'wf-obs-nb-001'

obs_svc = ObservabilityService(
    observability_repo=ObservabilityRepository(TEMP_DB),
    step_repo=StepRepository(TEMP_DB),
    decision_repo=DecisionRepository(TEMP_DB),
    security_repo=SecurityRepository(TEMP_DB),
)

# First, create the workflow run so FK constraints don't fail
from app.repositories.workflow_repository import WorkflowRepository
from app.state.workflow_state import WorkflowState, WorkflowStatus
now = utcnow_iso()
wf_state = WorkflowState(
    workflow_id=OBS_WF_ID, workflow_type='job_search',
    status=WorkflowStatus.RUNNING,
    current_step=WorkflowStep.SCORING,
    created_at=now, updated_at=now,
)
WorkflowRepository(TEMP_DB).create(OBS_WF_ID, 'job_search', wf_state.model_dump())
# Also create a run_metrics row so finalize_run_metrics can update it
ObservabilityRepository(TEMP_DB).create_run_metrics(str(uuid.uuid4()), OBS_WF_ID, now)

# Agent started
event_id = obs_svc.log_agent_started(OBS_WF_ID, 'scoring_agent', '3 jobs, resume res-001')
assert len(event_id) == 36
print(f'log_agent_started → event_id={event_id}')

# Agent completed
obs_svc.log_agent_completed(OBS_WF_ID, 'scoring_agent', event_id, '3 scores produced', 420)

# LLM call
obs_svc.log_llm_call(
    OBS_WF_ID, 'scoring_agent', 'anthropic', 'claude-haiku-4-5-20251001',
    tokens_input=1800, tokens_output=280, cost_usd=0.00038, latency_ms=420,
)

# Step lifecycle
step_id = obs_svc.log_step_started(OBS_WF_ID, WorkflowStep.SCORING)
assert len(step_id) == 36
obs_svc.log_step_completed(OBS_WF_ID, step_id, duration_ms=420, notes='3 jobs scored')
print(f'log_step_started  → step_id={step_id}')

# Security event
obs_svc.log_security_event(
    OBS_WF_ID, 'prompt_injection_attempt', 'high',
    'Instruction detected in job description: "ignore previous instructions"'
)

# Finalize metrics
obs_svc.finalize_run_metrics(
    OBS_WF_ID, total_llm_calls=1, total_tokens_input=1800,
    total_tokens_output=280, total_cost_usd=0.00038,
    total_duration_ms=1200, completed_at=utcnow_iso(),
)

# Verify from DB
with get_connection(TEMP_DB) as conn:
    events = conn.execute(
        'SELECT event_type FROM agent_events WHERE workflow_run_id=?', (OBS_WF_ID,)
    ).fetchall()
    llm_calls = conn.execute(
        'SELECT tokens_input FROM llm_calls WHERE workflow_run_id=?', (OBS_WF_ID,)
    ).fetchall()
    step_rows = conn.execute(
        'SELECT step, status FROM step_executions WHERE workflow_run_id=?', (OBS_WF_ID,)
    ).fetchall()
    sec_events = conn.execute(
        'SELECT severity FROM security_events WHERE workflow_run_id=?', (OBS_WF_ID,)
    ).fetchall()
    metrics = conn.execute(
        'SELECT total_llm_calls, total_cost FROM run_metrics WHERE workflow_run_id=?', (OBS_WF_ID,)
    ).fetchone()

event_types = [r[0] for r in events]
assert 'started' in event_types
assert 'completed' in event_types
assert llm_calls[0][0] == 1800
assert step_rows[0][0] == 'scoring'
assert step_rows[0][1] == 'completed'
assert sec_events[0][0] == 'high'
assert metrics[0] == 1  # total_llm_calls

print(f'\nDB verification:')
print(f'  agent_events : {event_types}')
print(f'  llm_calls    : tokens_input={llm_calls[0][0]}')
print(f'  step         : {step_rows[0][0]} / {step_rows[0][1]}')
print(f'  security     : severity={sec_events[0][0]}')
print(f'  run_metrics  : total_llm_calls={metrics[0]}, cost=${metrics[1]:.5f}')

# Failure isolation — repo crash must not propagate
broken_obs = MagicMock()
broken_obs.create_agent_event.side_effect = RuntimeError('DB locked')
broken_svc = ObservabilityService(
    observability_repo=broken_obs,
    step_repo=StepRepository(TEMP_DB),
    decision_repo=DecisionRepository(TEMP_DB),
    security_repo=SecurityRepository(TEMP_DB),
)
broken_svc.log_agent_started('wf-broken', 'agent', 'input')  # must not raise
print(f'\n✓ Repo failure swallowed — workflow not disrupted')

print('\n✓ ObservabilityService validated')

---
## 9. ReportGenerator — Run Summary and Job Report

**Purpose:** Verify that `ReportGenerator` assembles correct Markdown from repository data
and saves it to the `reports` table.

**Why reading from repos (not WorkflowState):** Reports may be regenerated after a run
completes. `WorkflowState` only exists in memory during a live run — the DB is the durable
source for report generation.

**What it proves:**
- `generate_run_summary()` builds a Markdown table from scored jobs and persists it
- `generate_job_report()` includes all five score dimensions
- Missing sections (no tailoring, no interview prep) are omitted gracefully — no KeyError

In [ ]:
import json

from app.repositories.advice_repository import AdviceRepository
from app.repositories.job_repository import JobRepository
from app.repositories.report_repository import ReportRepository
from app.repositories.review_repository import ReviewRepository
from app.repositories.score_repository import ScoreRepository
from app.repositories.tailoring_repository import TailoringRepository
from app.services.report_generator import ReportGenerator

RPT_WF_ID = 'wf-rpt-nb-001'
RPT_JOB_ID = 'job-rpt-001'

job_repo = JobRepository(TEMP_DB)
score_repo = ScoreRepository(TEMP_DB)
report_repo = ReportRepository(TEMP_DB)

# Seed: workflow run
from app.repositories.workflow_repository import WorkflowRepository
now = utcnow_iso()
wf_state = WorkflowState(
    workflow_id=RPT_WF_ID, workflow_type='job_search',
    status=WorkflowStatus.RUNNING,
    current_step=WorkflowStep.SCORING,
    created_at=now, updated_at=now,
)
WorkflowRepository(TEMP_DB).create(RPT_WF_ID, 'job_search', wf_state.model_dump())

# Seed: job
job_repo.upsert({
    'id': RPT_JOB_ID, 'source': 'linkedin',
    'title': 'Staff Engineer', 'company': 'TechCorp',
    'url': 'https://linkedin.com/jobs/rpt-001',
    'job_description': 'Staff Engineer role at TechCorp.',
    'normalized': {},
})

# Seed: score
score_data = {
    'overall_score': 88, 'technical_score': 85,
    'architecture_score': 90, 'leadership_score': 82,
    'domain_score': 80,
    'match_summary': 'Strong technical fit. Architecture skills stand out.',
    'strengths': ['Python expertise', 'Kubernetes at scale', 'GCP migrations'],
    'gaps': ['Limited fintech domain experience'],
    'recommended_next_action': 'shortlist_for_deep_review',
    'confidence': 90,
}
score_repo.create(
    str(uuid.uuid4()), RPT_WF_ID, RPT_JOB_ID, 'res-nb-001', score_data
)

# Build report generator
rg = ReportGenerator(
    score_repo=score_repo,
    review_repo=ReviewRepository(TEMP_DB),
    advice_repo=AdviceRepository(TEMP_DB),
    tailoring_repo=TailoringRepository(TEMP_DB),
    report_repo=report_repo,
    job_repo=job_repo,
)

# generate_run_summary
summary = rg.generate_run_summary(RPT_WF_ID)
assert RPT_WF_ID in summary
assert 'Jobs Scored' in summary
assert 'Staff Engineer' in summary
assert 'TechCorp' in summary

# Confirm persisted
saved = report_repo.get_by_run(RPT_WF_ID)
assert saved is not None
assert RPT_WF_ID in saved['report_markdown']

print('generate_run_summary() — first 600 chars:')
print('-' * 60)
print(summary[:600])
print('-' * 60)

# generate_job_report
job_report = rg.generate_job_report(RPT_WF_ID, RPT_JOB_ID)
assert 'Overall' in job_report
assert 'Technical' in job_report
assert 'Architecture' in job_report
assert 'Leadership' in job_report
assert 'Strong technical fit' in job_report
assert 'Tailoring' not in job_report  # no tailoring data → section omitted

print(f'\ngenerate_job_report() — {len(job_report)} chars, Tailoring section absent ✓')

# No data → empty string
empty_report = rg.generate_job_report(RPT_WF_ID, 'job-nonexistent')
assert empty_report == ''
print(f'generate_job_report() with no data → empty string ✓')

print('\n✓ ReportGenerator validated')

---
## 10. Cleanup

Deletes the temporary test database created in Section 1.

In [ ]:
if TEMP_DB.exists():
    TEMP_DB.unlink()
    print(f'Deleted test database: {TEMP_DB}')

print()
print('=' * 62)
print('  PHASE 2 SERVICES — VALIDATION COMPLETE')
print('=' * 62)
print()
print('Verified interactively:')
print('  ✓ JobPosting — work_mode default, SalaryInfo None fields')
print('  ✓ ResumeProfile — raw_text mandatory, ExperienceEntry.end_year=None')
print('  ✓ SkillNormalizer — alias lookup, deduplication, unknown passthrough')
print('  ✓ StatusManager — all workflow transitions, all job transitions')
print('  ✓ StatusManager — terminal states locked, InvalidTransitionError with context')
print('  ✓ JobDiscoveryService — normalize, deduplicate, cap, scraper failure resilience')
print('  ✓ ResumeParser — heuristic parse, cache hit/miss, enhance_fn mock, empty error')
print('  ✓ ObservabilityService — agent events, LLM calls, step lifecycle, security events')
print('  ✓ ObservabilityService — repo failure swallowed, workflow not disrupted')
print('  ✓ ReportGenerator — run summary, job report, missing sections omitted')